In my point of view Instruction finetuning without masking gives better output.

In [1]:
!pip install transformers bitsandbytes peft accelerate datasets PyMuPDF -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.7 MB/s eta 0:00:00


In [2]:
from datasets import Dataset

In [3]:
import fitz

text_blocks = []
path = "/content/New Microsoft Word Document.docx"
with fitz.open(path) as pdf:
  for page in pdf:
    text = page.get_text("text")
    text_blocks.append(text)

In [4]:
import re

def split_paragraphs(pages):
  chunks = []
  for page_text in pages:
    question_answer_pairs = re.split(r"Q: ", page_text)
    for question_answer_pair in question_answer_pairs:
      if not question_answer_pair.strip():
        continue
      question = question_answer_pair.split("A: ")[0]
      answer = question_answer_pair.split("A: ")[1]
      combined = f"Question: {question.strip()}\nAnswer: {answer.strip()}"
      chunks.append(combined)

  return chunks

In [5]:
chunks = split_paragraphs(text_blocks)

In [6]:
data = [{'text': c} for c in chunks]

In [7]:
dataset = Dataset.from_list(data)

In [8]:
dataset

Dataset({
    features: ['text'],
    num_rows: 100
})

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

In [10]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [12]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [13]:
def tokenize_fn(examples):
    results = {"input_ids": [], "attention_mask": [], "labels": []}

    for text in examples["text"]:
        # Full sequence tokenization
        full = tokenizer(text, truncation=True, padding="max_length", max_length=512)
        input_ids = full["input_ids"]

        # Find where the answer starts by tokenizing just the question part
        # text format is: "Question: ...\nAnswer: ..."
        question_part = text.split("\nAnswer:")[0] + "\nAnswer:"
        question_tokens = tokenizer(question_part, add_special_tokens=False)["input_ids"]
        question_len = len(question_tokens)

        # Build labels: mask question tokens with -100, keep answer tokens
        labels = [-100] * question_len + input_ids[question_len:]

        # Ensure labels same length as input_ids (due to padding)
        labels = labels[:512]
        labels += [-100] * (512 - len(labels))  # pad remainder with -100

        results["input_ids"].append(input_ids)
        results["attention_mask"].append(full["attention_mask"])
        results["labels"].append(labels)

    return results

In [14]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [16]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [17]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [18]:
q_lora_model = get_peft_model(model, lora_config)

In [19]:
training_args = TrainingArguments(
    output_dir="/content/trained_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    report_to="none")

In [20]:
trainer = Trainer(
    model = q_lora_model,
    args = training_args,
    train_dataset = tokenized
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,5.815175
100,2.895989


TrainOutput(global_step=100, training_loss=4.355582275390625, metrics={'train_runtime': 167.8294, 'train_samples_per_second': 1.192, 'train_steps_per_second': 0.596, 'total_flos': 636296468889600.0, 'train_loss': 4.355582275390625, 'epoch': 2.0})

In [22]:
import shutil
from google.colab import files

# shutil.make_archive(output_name, format, folder_to_zip)
shutil.make_archive('/content/instructional_model', 'zip', '/content/trained_model/checkpoint-100')

files.download('/content/instructional_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
model_path = "/content/trained_model/checkpoint-100"

In [24]:
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [25]:
prompt = "How to reach the non-target customers"

In [26]:
formatted_prompt = f"Question: {prompt}\nAnswer: "

In [27]:
input = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

In [30]:
outputs = model.generate(
    **input,
    max_new_tokens=200,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

In [31]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

"Question: How to reach the non-target customers\nAnswer: 1. Identify your target customers and understand their behavior patterns. 2. Define specific needs of the target customers that you want to address through the email campaign. 3. Research on the competitors' activities and identify similar groups or segments that they have ignored, and use this information to create a customized content for the email campaign. 4. Target your emails to a list with a high open rate, low click-through rate, and low bounce rate, which will improve the success rate of the email campaign."